# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df_features = pd.read_csv('../../src/data/day-of-week-not-scaled.csv')


In [3]:
df_target = pd.read_csv('../../src/data/dayofweek.csv')
y = df_target['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(
    df_features, y, test_size=0.2, random_state=21, stratify=y
)


## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [4]:
svm = SVC(random_state=21, probability=True)
svm_params = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': [None, 'balanced']
}
gs_svm = GridSearchCV(svm, svm_params, cv=5, scoring='accuracy', n_jobs=-1)
gs_svm.fit(X_train, y_train)

print('Best params:', gs_svm.best_params_)
print('Best score:', gs_svm.best_score_)


Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
Best score: 0.8761090458488228


In [5]:
df_svm = pd.DataFrame(gs_svm.cv_results_)
df_svm_sorted = df_svm.sort_values('rank_test_score')
df_svm_sorted[['param_kernel', 'param_C', 'param_gamma', 'param_class_weight', 'mean_test_score', 'rank_test_score']].head(10)


,param_kernel,param_C,param_gamma,param_class_weight,mean_test_score,rank_test_score
64,rbf,10.0,auto,NaN,0.876109,1
70,rbf,10.0,auto,balanced,0.863500,2
52,rbf,5.0,auto,NaN,0.816018,3
58,rbf,5.0,auto,balanced,0.807865,4
69,linear,10.0,auto,balanced,0.721052,5
66,linear,10.0,scale,balanced,0.721052,5
63,linear,10.0,auto,NaN,0.719587,7
60,linear,10.0,scale,NaN,0.719587,7
57,linear,5.0,auto,balanced,0.706234,9
54,linear,5.0,scale,balanced,0.706234,9


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [6]:
dt = DecisionTreeClassifier(random_state=21)
dt_params = {
    'max_depth': range(1, 50),
    'class_weight': [None, 'balanced'],
    'criterion': ['gini', 'entropy']
}
gs_dt = GridSearchCV(dt, dt_params, cv=5, scoring='accuracy', n_jobs=-1)
gs_dt.fit(X_train, y_train)

print('Best params:', gs_dt.best_params_)
print('Best score:', gs_dt.best_score_)


Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 23}
Best score: 0.8738592867960897


In [7]:
df_dt = pd.DataFrame(gs_dt.cv_results_)
df_dt_sorted = df_dt.sort_values('rank_test_score')
df_dt_sorted[['param_max_depth', 'param_class_weight', 'param_criterion', 'mean_test_score', 'rank_test_score']].head(10)


,param_max_depth,param_class_weight,param_criterion,mean_test_score,rank_test_score
131,34,balanced,gini,0.873859,1
142,45,balanced,gini,0.873859,1
141,44,balanced,gini,0.873859,1
140,43,balanced,gini,0.873859,1
139,42,balanced,gini,0.873859,1
138,41,balanced,gini,0.873859,1
137,40,balanced,gini,0.873859,1
136,39,balanced,gini,0.873859,1
135,38,balanced,gini,0.873859,1
145,48,balanced,gini,0.873859,1


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [8]:
rf = RandomForestClassifier(random_state=21)
rf_params = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': range(1, 50),
    'class_weight': [None, 'balanced'],
    'criterion': ['gini', 'entropy']
}
gs_rf = GridSearchCV(rf, rf_params, cv=5, scoring='accuracy', n_jobs=-1)
gs_rf.fit(X_train, y_train)

print('Best params:', gs_rf.best_params_)
print('Best score:', gs_rf.best_score_)


Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 28, 'n_estimators': 50}
Best score: 0.9042902381935839


In [9]:
df_rf = pd.DataFrame(gs_rf.cv_results_)
df_rf_sorted = df_rf.sort_values('rank_test_score')
df_rf_sorted[['param_n_estimators', 'param_max_depth', 'param_class_weight', 'param_criterion', 'mean_test_score', 'rank_test_score']].head(10)


,param_n_estimators,param_max_depth,param_class_weight,param_criterion,mean_test_score,rank_test_score
110,50,28,NaN,gini,0.904290,1
123,100,31,NaN,gini,0.904287,2
530,50,35,balanced,gini,0.903549,3
538,50,37,balanced,gini,0.903549,3
562,50,43,balanced,gini,0.903549,3
542,50,38,balanced,gini,0.903549,3
570,50,45,balanced,gini,0.903549,3
554,50,41,balanced,gini,0.903549,3
566,50,44,balanced,gini,0.903549,3
578,50,47,balanced,gini,0.903549,3


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [10]:
from tqdm.notebook import tqdm

results = []
n_estimators_options = [5, 10, 50, 100]
max_depth_options = range(1, 50)
class_weight_options = [None, 'balanced']
criterion_options = ['gini', 'entropy']

total = len(n_estimators_options) * len(max_depth_options) * len(class_weight_options) * len(criterion_options)

with tqdm(total=total) as pbar:
    for n in n_estimators_options:
        for d in max_depth_options:
            for cw in class_weight_options:
                for crit in criterion_options:
                    model = RandomForestClassifier(
                        n_estimators=n, max_depth=d,
                        class_weight=cw, criterion=crit,
                        random_state=21
                    )
                    scores = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
                    results.append({
                        'n_estimators': n,
                        'max_depth': d,
                        'class_weight': str(cw),
                        'criterion': crit,
                        'mean_accuracy': scores.mean(),
                        'std_accuracy': scores.std()
                    })
                    pbar.update(1)


  0%|          | 0/784 [00:00<?, ?it/s]

In [11]:
df_manual = pd.DataFrame(results)
df_manual_sorted = df_manual.sort_values('mean_accuracy', ascending=False)
df_manual_sorted.head(10)


,n_estimators,max_depth,class_weight,criterion,mean_accuracy,std_accuracy
500,50,28,None,gini,0.904290,0.010961
708,100,31,None,gini,0.904287,0.015204
582,50,48,balanced,gini,0.903549,0.012503
546,50,39,balanced,gini,0.903549,0.012503
542,50,38,balanced,gini,0.903549,0.012503
554,50,41,balanced,gini,0.903549,0.012503
558,50,42,balanced,gini,0.903549,0.012503
550,50,40,balanced,gini,0.903549,0.012503
562,50,43,balanced,gini,0.903549,0.012503
530,50,35,balanced,gini,0.903549,0.012503


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [12]:
best_model = gs_rf.best_estimator_
y_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {test_acc:.5f}')


Test accuracy: 0.92899
